In [1]:
import os
import re
import json
from datetime import datetime
import random
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

# 1. Test Dataset Generation from Documents

In [3]:
from evaluation.eval import prepare_testset_documents, generate_qa_dataset, generate_rag_responses

In [4]:
docs = prepare_testset_documents("eval_data")

Successfully parsed MSFT-anual-report.html
Successfully parsed amd-20251227.html
Successfully parsed amzn-20251231.html
Successfully parsed chtr-20251231.html
Successfully parsed goog-20251231.html
Successfully parsed meta-20251231.html
Successfully parsed mstr-20251231.html
Successfully parsed nvda-20260125.html
Successfully parsed ptrn-20251231.html
Successfully parsed Excise Taxes.pdf
Successfully parsed Medical and Dental Expenses.pdf
Successfully parsed Miscellaneous Deductions.pdf
Successfully parsed Reporting Cash Payments of Over $10,000.pdf
Successfully parsed Selling Your Home.pdf
Successfully parsed Tax Information for Homeowners.pdf
Successfully parsed Tax Withholding and Estimated Tax.pdf
Successfully parsed Taxable and Nontaxable Income.pdf
Successfully parsed U.S. Tax Guide for Aliens.pdf
Successfully parsed Your Rights As A Taxpayer.pdf
Successfully parsed AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf
Successfully pa

In [35]:
from rag_pipeline.query_engine import vectorstore

In [38]:
vectorstore_db = vectorstore(persist_directory="/Volumes/LaCie/Projects_portfolio/IntelliQA/chroma_db", 
                                     documents=docs)

Adding documents to existing vector space


# 2. Synthetic QA Dataset Generation

In [39]:
# imports for synthetic QA Dataset Generation
from ragas import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama # ollama can run locally
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
# csv files are temporarily filtered out because they are large and dominate the dataset
gen_docs = [file for file in docs if not file.metadata['filename'].endswith('.csv')]

# subsample documents to control cost and runtime during synthetic QA generation
random_docs = random.sample(gen_docs, 200)

run_config = RunConfig(max_workers=2, timeout=180)

In [ ]:
generator_llm = LangchainLLMWrapper(
                ChatOllama(
                    model="qwen2.5",
                    temperature=0,
                    # model_kwargs={"response_format":{"type":"json_object"}}
                    format="json"
                    )
                )

generator_embeddings = LangchainEmbeddingsWrapper(
                HuggingFaceEmbeddings(
                    model_name="sentence-transformers/all-MiniLM-L6-v2"
                    )
                )           

In [ ]:
testset = generate_qa_dataset(
                                random_docs, generator_llm, generator_embeddings,  
                                run_config=run_config,
                                testset_size=100
                            )

In [ ]:
df = testset.to_pandas()

In [ ]:
testset.to_pandas().to_json("test_data/rag_evaluation_testset.json", orient="records", indent=2)

# 3. RAG Pipeline Execution

In [40]:
df = pd.read_json("test_data/rag_evaluation_testset.json")

In [207]:
vectorstore_db = vectorstore('chroma_backup/chroma_db')

Loading existing vector space


In [208]:
df_32 = pd.read_json("test_data/curated_rag_evaluation_testset_32.jsonl", orient="records", lines=True)

In [209]:
generate_rag_responses(df_32, vectorstore_db, session_id="test_session")

Row 1 processed and saved.
Row 2 processed and saved.
Row 3 processed and saved.
Row 4 processed and saved.
Row 5 processed and saved.
Row 6 processed and saved.
Row 7 processed and saved.
Row 8 processed and saved.
Row 9 processed and saved.
Row 10 processed and saved.
Row 11 processed and saved.
Row 12 processed and saved.
Row 13 processed and saved.
Row 14 processed and saved.
Row 15 processed and saved.
Row 16 processed and saved.
Row 17 processed and saved.
Row 18 processed and saved.
Row 19 processed and saved.
Row 20 processed and saved.
Row 21 processed and saved.
Row 22 processed and saved.
Row 23 processed and saved.
Row 24 processed and saved.
Row 25 processed and saved.
Row 26 processed and saved.
Row 27 processed and saved.
Row 28 processed and saved.
Row 29 processed and saved.
Row 30 processed and saved.
Row 31 processed and saved.
Row 32 processed and saved.


In [189]:
with open("test_data/rag_results.jsonl", "r") as f:
    rag_results = [json.loads(line) for line in f]

In [ ]:
# print(rag_results[-1]['user_input'])
print(df.iloc[31].user_input)

# 4. RAG Evaluation

In [198]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    ResponseRelevancy,
)
from langchain_google_genai import ChatGoogleGenerativeAI

In [199]:
evaluator_llm = LangchainLLMWrapper(
                    ChatGoogleGenerativeAI(
                        model="gemini-flash-lite-latest",
                        temperature=0,
                        # model_kwargs={"response_format": {"type": "json_object"}},
                    )
                )
evaluator_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 921.06it/s]


In [200]:
metrics = [
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

run_config = RunConfig(max_workers=2, timeout=180)

In [201]:
result = evaluate(dataset=EvaluationDataset.from_list(rag_results), metrics=metrics, 
                   run_config=run_config, raise_exceptions=True)

Evaluating:   0%|          | 0/128 [03:00<?, ?it/s]


TimeoutError: 

In [ ]:
result_df = result.to_pandas()

In [49]:
result_df = pd.read_csv("evaluation/rag_eval_v2.csv")

In [50]:
recall_fails = result_df[result_df['context_recall']==0][['user_input','retrieved_contexts','response', 'reference']]

In [ ]:
print(result_df['reference'].iloc[1])

In [ ]:
print(result_df['user_input'].iloc[1])

In [ ]:
print(result_df['response'].iloc[1])

In [ ]:
for r in result_df['retrieved_contexts'].iloc[1]:
    print(r)

In [ ]:
from IPython.display import display, HTML

subset = result_df[result_df['context_recall'] == 0][
    ['user_input', 'retrieved_contexts', 'response', 'reference']
]

display(HTML(subset.to_html().replace('<td>', '<td style="text-align:left; white-space:pre-wrap; word-wrap:break-word; max-width:400px;">')))